In [ ]:
import os
import json

import torch
import torch.nn as nn

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support
)

from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

from utils.dataset import get_dataloaders
from utils.models import get_model

In [ ]:

DATASET_DIR = "data"

# ResNet and VGG learn well when trained from scratch
# DenseNet and EfficientNet did not learn meaningfull representations and majority-collapsed
# A future add-on may consider using weighed sampling or focal loss
MODEL_NAME = "resnet50"
BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-3

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

In [ ]:
def train_epoch(model, loader, criterion, optimizer):

    model.train()

    running_loss = 0

    for images, labels in loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)

In [ ]:
@torch.no_grad()
def evaluate(model, loader):

    model.eval()

    preds = []
    targets = []

    for images, labels in loader:

        images = images.to(DEVICE)

        outputs = model(images)

        pred = outputs.argmax(1)

        preds.extend(pred.cpu().numpy())
        targets.extend(labels.numpy())

    return preds, targets

In [ ]:

def main():

    os.makedirs("checkpoints", exist_ok=True)
    os.makedirs("outputs", exist_ok=True)

    (
        train_loader,
        val_loader,
        test_loader,
        class_names
    ) = get_dataloaders(
        DATASET_DIR,
        batch_size=BATCH_SIZE
    )

    model = get_model(
        MODEL_NAME,
        len(class_names),
        pretrained=False
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss()

    optimizer = AdamW(
        model.parameters(),
        lr=LR
    )

    scheduler = ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2
    )

    best_val_acc = 0

    for epoch in range(EPOCHS):

        train_loss = train_epoch(
            model,
            train_loader,
            criterion,
            optimizer
        )

        preds, targets = evaluate(
            model,
            val_loader
        )

        val_acc = accuracy_score(
            targets,
            preds
        )

        scheduler.step(train_loss)

        print(
            f"Epoch {epoch+1}/{EPOCHS}"
            f" | Loss={train_loss:.4f}"
            f" | Val Acc={val_acc:.4f}"
        )

        if val_acc > best_val_acc:

            best_val_acc = val_acc

            torch.save(
                model.state_dict(),
                f"checkpoints/{MODEL_NAME}_best.pth"
            )

    print("Evaluating on test set")

    model.load_state_dict(
        torch.load(
            f"checkpoints/{MODEL_NAME}_best.pth",
            weights_only=False
        )
    )

    preds, targets = evaluate(
        model,
        test_loader
    )

    acc = accuracy_score(
        targets,
        preds
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            targets,
            preds,
            average="weighted"
        )
    )

    cm = confusion_matrix(
        targets,
        preds
    )

    report = classification_report(
        targets,
        preds,
        target_names=class_names
    )

    print(report)

    metrics = {
        "accuracy": float(acc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1)
    }

    with open(
        f"outputs/{MODEL_NAME}_metrics.json",
        "w"
    ) as f:
        json.dump(metrics, f, indent=4)

    with open(
        f"outputs/{MODEL_NAME}_report.txt",
        "w"
    ) as f:
        f.write(report)

    with open(f"outputs/{MODEL_NAME}_confusion_matrix.txt", "w") as f:
        f.write(str(cm))

if __name__ == "__main__":
    main()
